
<a id="top"></a>
# Modulo 4 — MLflow per il tracciamento degli esperimenti

## Indice

- [Introduzione a MLflow](#intro-mlflow)
- [Setup e caricamento dati](#setup)
- [Feature engineering con pipeline scikit-learn](#pipeline)
- [MLflow: configurazione e autolog](#mlflow-config)
- [Training e confronto di più modelli](#training)
- [Visualizzazione con MLflow UI](#mlflow-ui)
- [Model Registry e inferenza](#registry)



<a id="intro-mlflow"></a>
## Introduzione a MLflow

Quando si lavora con modelli di machine learning, una delle sfide principali è tenere traccia di:
- quali modelli sono stati addestrati
- con quali parametri
- quali performance hanno ottenuto
- dove sono salvati gli artefatti (grafici, modelli serializzati)

**MLflow** è uno strumento open-source per gestire il ciclo di vita del modello. Permette di:

1. **Tracking**: registrare parametri, metriche e artefatti di ogni esperimento
2. **Model Registry**: salvare e versionare modelli pronti per produzione
3. **Deployment**: caricare modelli in modo standard per inferenza

In questo notebook useremo MLflow per tracciare l'addestramento di diversi modelli sul dataset Bank Marketing, confrontare le performance e registrare il migliore per utilizzo futuro.



<a id="setup"></a>
## Setup e caricamento dati

Importiamo le librerie necessarie e carichiamo il dataset Bank Marketing da UCI.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ucimlrepo import fetch_ucirepo

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, RobustScaler
from sklearn.inspection import permutation_importance

import mlflow
import mlflow.sklearn

# Caricamento dataset Bank Marketing
bank_marketing = fetch_ucirepo(id=222)
X_raw = bank_marketing.data.features.copy()
y_raw = bank_marketing.data.targets.copy()

# Sample di 10k righe per velocizzare
X_raw, _, y_raw, _ = train_test_split(
    X_raw, y_raw, 
    train_size=10000, 
    random_state=42, 
    stratify=y_raw
)

print(f"Dataset: {X_raw.shape[0]} righe, {X_raw.shape[1]} colonne")
print(f"Target: {y_raw.value_counts().to_dict()}")
X_raw.head()



<a id="pipeline"></a>
## Feature engineering con pipeline scikit-learn

Prepariamo i dati usando lo stesso approccio visto in T3: feature selection e pipeline modulare con `ColumnTransformer`.

Questo approccio garantisce:
- **Consistenza**: stesso preprocessing in train e test
- **Riproducibilità**: l'intera pipeline è serializzabile
- **Modularità**: possiamo testare diversi modelli con lo stesso preprocessor


In [ ]:
# Selezione feature (escludo duration per evitare leakage)
binary_cols = ["default", "housing", "loan"]
nominal_cols = ["job", "marital", "contact", "poutcome"]
ordinal_cols = ["education", "month"]
numeric_cols = ["age", "balance", "campaign", "pdays", "previous"]

feature_cols = binary_cols + nominal_cols + ordinal_cols + numeric_cols

# Target mapping
df = X_raw[feature_cols].copy()
y = y_raw.iloc[:, 0].map({"no": 0, "yes": 1})
X = df

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Class balance: {y_train.value_counts(normalize=True).to_dict()}")


Costruiamo la pipeline di preprocessing usando esattamente la stessa struttura di T3.

La pipeline usa `RobustScaler` per le variabili numeriche (robusto agli outlier) e gestisce separatamente:
- **Binary**: encoding ordinale yes/no → 0/1
- **Ordinal**: encoding con ordine semantico (education, month)
- **Nominal**: one-hot encoding (drop first per evitare multicollinearità)
- **Numeric**: scaling robusto


In [ ]:
# Binary encoder
binary_encoder = OrdinalEncoder(
    categories=[["no", "yes"]] * len(binary_cols),
    dtype=float
)

# Ordinal encoder
education_order = ["primary", "secondary", "tertiary"]
month_order = ["jan", "feb", "mar", "apr", "may", "jun",
               "jul", "aug", "sep", "oct", "nov", "dec"]
ordinal_encoder = OrdinalEncoder(
    categories=[education_order, month_order],
    handle_unknown="use_encoded_value",
    unknown_value=-1,
    dtype=float
)

# Nominal encoder
nominal_encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False,
    drop="first"
)

# Numeric scaler (RobustScaler per gestire outlier)
numeric_scaler = RobustScaler()

# ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ("bin", binary_encoder, binary_cols),
        ("ord", ordinal_encoder, ordinal_cols),
        ("nom", nominal_encoder, nominal_cols),
        ("num", numeric_scaler, numeric_cols)
    ]
)

print("Preprocessor creato con successo")
print(f"  Binary: {len(binary_cols)} colonne")
print(f"  Ordinal: {len(ordinal_cols)} colonne")
print(f"  Nominal: {len(nominal_cols)} colonne")
print(f"  Numeric: {len(numeric_cols)} colonne")


In [ ]:
# Definizione dei modelli da confrontare
models = {
    'RandomForest': Pipeline([
        ('preprocess', preprocessor),
        ('model', RandomForestClassifier(
            n_estimators=200,
            max_depth=10,
            min_samples_split=20,
            class_weight='balanced',
            random_state=42,
            n_jobs=-1
        ))
    ]),
    
    'LogisticRegression': Pipeline([
        ('preprocess', preprocessor),
        ('model', LogisticRegression(
            C=0.1,
            class_weight='balanced',
            solver='liblinear',
            random_state=42,
            max_iter=1000
        ))
    ]),
    
    'XGBoost': Pipeline([
        ('preprocess', preprocessor),
        ('model', XGBClassifier(
            n_estimators=200,
            max_depth=6,
            learning_rate=0.1,
            random_state=42,
            n_jobs=-1,
            eval_metric='logloss'
        ))
    ]),
    
    'SVM': Pipeline([
        ('preprocess', preprocessor),
        ('model', SVC(
            C=1.0,
            kernel='rbf',
            class_weight='balanced',
            probability=True,
            random_state=42
        ))
    ])
}

print(f"Creati {len(models)} modelli con pipeline dedicate:")
for name in models.keys():
    print(f"  - {name}")



<a id="mlflow-config"></a>
## MLflow: configurazione e autolog

Prima di addestrare i modelli, configuriamo MLflow per tracciare automaticamente parametri, metriche e artefatti.

**Concetti chiave MLflow:**

1. **Tracking URI**: dove salvare i dati degli esperimenti (locale: `file:../mlruns`)
2. **Experiment**: contenitore logico per raggruppare run correlate
3. **Autolog**: tracciamento automatico di parametri e metriche per modelli scikit-learn
4. **Run**: singola esecuzione di training (parametri + metriche + artefatti)


In [ ]:
# Configurazione MLflow
import os

# Opt-in necessario nelle versioni recenti per usare il backend file legacy.
os.environ.setdefault("MLFLOW_ALLOW_FILE_STORE", "true")

mlflow.set_tracking_uri("file:../mlruns")
mlflow.set_experiment("bank_marketing_modulo4")

# Autolog per scikit-learn (traccia parametri e metriche automaticamente)
mlflow.sklearn.autolog(
    log_input_examples=True,
    log_model_signatures=True,
    log_models=False
)

print("MLflow configurato:")
print(f"  Tracking URI: {mlflow.get_tracking_uri()}")
print(f"  Experiment: bank_marketing_modulo4")
print(f"  Autolog attivo per scikit-learn")

In [ ]:
import tempfile
import os
from mlflow.models import infer_signature

def train_and_log_model(model_name, pipeline, X_train, y_train, X_test, y_test):
    """
    Addestra un modello e traccia metriche, tag e artefatti con MLflow.
    
    Tracciamento MLflow:
    - Metriche: accuracy, precision, recall, F1, ROC-AUC
    - Tag: dataset, model_type, author
    - Artefatti: Permutation Feature Importance (PFI)
    - Modello: pipeline completa con signature e input_example
    """
    with mlflow.start_run(run_name=model_name):
        
        # 1. Training
        print(f"\nTraining {model_name}...")
        pipeline.fit(X_train, y_train)
        
        # 2. Predizioni
        y_pred = pipeline.predict(X_test)
        y_proba = pipeline.predict_proba(X_test)[:, 1]
        
        # 3. Metriche
        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred)
        rec = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        auc = roc_auc_score(y_test, y_proba)
        
        # Log metriche custom
        mlflow.log_metric("test_accuracy", acc)
        mlflow.log_metric("test_precision", prec)
        mlflow.log_metric("test_recall", rec)
        mlflow.log_metric("test_f1", f1)
        mlflow.log_metric("test_roc_auc", auc)
        
        print(f"  Accuracy: {acc:.4f}, ROC-AUC: {auc:.4f}")
        
        # 4. Tag personalizzati
        mlflow.set_tag("dataset", "bank_marketing")
        mlflow.set_tag("model_type", model_name)
        mlflow.set_tag("author", "corso_ai")
        
        # 5. Permutation Feature Importance (PFI)
        print("  Calcolando PFI...")
        pfi_result = permutation_importance(
            pipeline, X_test, y_test,
            n_repeats=10,
            random_state=42,
            n_jobs=-1
        )
        
        # Plot PFI top 15 features
        importances_df = pd.DataFrame({
            'feature': X_test.columns,
            'importance': pfi_result.importances_mean
        }).sort_values('importance', ascending=False).head(15)
        
        fig, ax = plt.subplots(figsize=(10, 6))
        ax.barh(importances_df['feature'][::-1], importances_df['importance'][::-1])
        ax.set_xlabel('Importance (drop in accuracy)')
        ax.set_title(f'Permutation Feature Importance - {model_name}')
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        
        # Salva e logga come artefatto
        with tempfile.NamedTemporaryFile(suffix='.png', delete=False) as f:
            fig.savefig(f.name, dpi=100, bbox_inches='tight')
            mlflow.log_artifact(f.name, artifact_path='explainability')
            os.unlink(f.name)
        plt.close(fig)
        
        # 6. Log modello con signature e input_example
        signature = infer_signature(X_train, y_train)
        input_example = X_train.head(3)
        mlflow.sklearn.log_model(
            pipeline,
            "model",
            signature=signature,
            input_example=input_example,
            serialization_format="cloudpickle"
        )
        
        print(f"  Run completata: {model_name}")
        
        return acc, prec, rec, f1, auc

print("Funzione train_and_log_model definita")


In [ ]:
# Training e confronto di più modelli

results = {}

for model_name, pipeline in models.items():
    metrics = train_and_log_model(
        model_name, 
        pipeline, 
        X_train, y_train, 
        X_test, y_test
    )
    results[model_name] = metrics

# Confronto risultati
results_df = pd.DataFrame(
    results, 
    index=['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
).T.sort_values('roc_auc', ascending=False)

print("\n" + "="*60)
print("CONFRONTO MODELLI")
print("="*60)
results_df.round(4)


<a id="mlflow-ui"></a>
## Visualizzazione con MLflow UI

Per visualizzare tutti gli esperimenti tracciati, lanciare MLflow UI da terminale.

### Istruzioni per lanciare MLflow UI

1. Apri il terminale di VSCode (Terminal → New Terminal)
2. Assicurati di essere nella **root del progetto** (la cartella `ai-course`)
3. Esegui il comando:

```bash
MLFLOW_ALLOW_FILE_STORE=true uv run mlflow ui --backend-store-uri file:./mlruns
```

4. Apri il browser all'indirizzo: http://127.0.0.1:5000  o alternativamente se il primo non funziona: http://localhost:5000
5. Seleziona l'esperimento `bank_marketing_modulo4`

💡 Su macOS la porta 5000 può essere occupata (AirPlay Receiver): in quel caso aggiungi `--port 5001`.

### Cosa puoi fare nella UI:

- **Confrontare metriche**: ordina le run per ROC-AUC, accuracy, F1
- **Visualizzare parametri**: ispeziona gli iperparametri di ogni modello
- **Esplorare artefatti**: scarica i plot PFI di ogni run
- **Confrontare run**: seleziona 2+ run e usa il bottone "Compare"


<a id="registry"></a>
## Model Registry e inferenza

Una volta identificato il modello migliore, possiamo registrarlo nel **Model Registry** di MLflow per versionarlo e riutilizzarlo.

### Passaggi per registrare un modello:

1. Nella MLflow UI, seleziona la run del modello migliore (es. quello con ROC-AUC più alto)
2. Nella pagina della run, sezione **"Logged models"** → clicca su `model`
3. Clicca sul bottone **"Register model"**
4. Inserisci il nome: `bank_marketing_model`
5. Clicca su "Register"

💡 In MLflow 3 il modello non compare più nella scheda "Artifacts" della run, ma nella sezione "Logged models".

### Assegnare l'alias "production":

1. Vai alla sezione **"Models"** (menu laterale MLflow UI)
2. Clicca sul modello `bank_marketing_model`
3. Clicca sulla versione (es. Version 1) → accanto ad "Aliases" clicca il pulsante per aggiungere un alias
4. Aggiungi l'alias: `production`
5. Conferma con "Save aliases"

A questo punto il modello è pronto per essere caricato in produzione usando:

```python
model = mlflow.sklearn.load_model("models:/bank_marketing_model@production")
```

Questo approccio garantisce:
- **Versioning**: ogni registrazione crea una nuova versione
- **Aliasing**: alias come `production`, `staging`, `champion` per identificare rapidamente il modello da usare
- **Tracciabilità**: ogni versione è collegata alla run originale con parametri e metriche


### Caricamento e inferenza

Una volta registrato, il modello può essere caricato da qualsiasi notebook o applicazione.


In [ ]:
# Caricamento del modello registrato
model_uri = "models:/bank_marketing_model@production"

try:
    loaded_pipeline = mlflow.sklearn.load_model(model_uri)
    print(f"Modello caricato: {model_uri}")
    
    # Test su sample del test set
    sample = X_test.head(10)
    predictions = loaded_pipeline.predict(sample)
    probabilities = loaded_pipeline.predict_proba(sample)[:, 1]
    
    # Risultati
    results = pd.DataFrame({
        'actual': y_test.head(10).values,
        'predicted': predictions,
        'proba_yes': probabilities.round(3),
        'age': sample['age'].values,
        'balance': sample['balance'].values
    })
    
    print("\nPredizioni su 10 clienti:")
    display(results)
    
    sample_acc = accuracy_score(results['actual'], results['predicted'])
    print(f"\nAccuracy sul campione: {sample_acc:.2%}")
    
except Exception as e:
    print(f"Errore: {e}")
    print("Assicurati di aver registrato il modello come 'bank_marketing_model' con alias 'production'")


## Test webapp con Streamlit

Ora che il modello è registrato, possiamo testare l'applicazione web che lo utilizza per fare predizioni in tempo reale.

### Configurazione webapp

Prima di lanciare l'app, verifica che il file di configurazione punti al modello corretto.

**File: `Webapp/config.py`**

```python
MLFLOW_TRACKING_URI = MLRUNS_DIR.as_uri()        # ← cartella mlruns nella root del progetto
MLFLOW_MODEL_NAME = "bank_marketing_model"      # ← Deve corrispondere al nome registrato
MLFLOW_MODEL_ALIAS = "production"                # ← Deve corrispondere all'alias assegnato
```

Puoi modificare questi valori per sperimentare con modelli diversi.

### Lancio webapp

1. Apri il terminale di VSCode
2. Assicurati di essere nella root del progetto (cartella `ai-course`)
3. Esegui:

```bash
cd Webapp
uv run streamlit run app_bank.py
```

4. L'app si aprirà nel browser (solitamente http://localhost:8501)
5. Inserisci i dati di un cliente e clicca su "Stima la probabilità di sottoscrizione"
6. L'app caricherà automaticamente il modello registrato e farà la predizione

### Come funziona

L'applicazione:
- Legge la configurazione da `config.py`
- Carica il modello da MLflow usando `models:/bank_marketing_model@production`
- Estrae automaticamente le feature richieste dalla pipeline
- Genera i controlli UI dinamicamente (selectbox per categorie, slider per numeri)
- Applica il preprocessing e la predizione con la pipeline completa

Questo dimostra come un modello tracciato con MLflow può essere utilizzato in produzione senza dover riscrivere il codice di preprocessing!

---

## Riepilogo

In questo notebook abbiamo imparato a usare **MLflow** per gestire il ciclo di vita del modello:

### 1. Tracking automatico con autolog
- `mlflow.sklearn.autolog()` traccia parametri e metriche automaticamente
- Funziona con tutti i modelli scikit-learn e pipeline

### 2. Logging personalizzato
- **Metriche custom**: `mlflow.log_metric()` per metriche specifiche del problema
- **Tag**: `mlflow.set_tag()` per metadata (dataset, author, model_type)
- **Artefatti**: `mlflow.log_artifact()` per salvare plot, file, report

### 3. Model signature e input example
- `infer_signature()`: definisce input/output schema del modello
- `input_example`: esempio di dati per testing e validazione
- Garantisce compatibilità e documentazione automatica

### 4. Model Registry
- Registrazione tramite UI: traccia versioni e metadata
- Aliasing: `production`, `staging` per gestire deployment
- Caricamento: `models:/model_name@alias` per inferenza

### 5. Riproducibilità e deployment
- La pipeline loggata contiene preprocessing + modello
- `pipeline.predict()` funziona su dati raw (preprocessing automatico)
- Ambiente, parametri e metriche tracciati per ogni run
- Webapp pronta per produzione con caricamento dinamico del modello

### Prossimi passi
- Esplora MLflow UI per confrontare modelli
- Registra il modello migliore nel Model Registry
- Prova a modificare `config.py` per caricare modelli diversi
- Nell'esercitazione applicherai questi concetti al dataset House Sales
